In [ ]:
import tensorflow as tf

In [ ]:
import sys
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
import scipy.misc

In [ ]:
sys.path.append('/root/Work/CogFashion/cftfmodels/')

In [ ]:
#from alexnet.base.models import conv
from alexnet.base.models import *
from alexnet.base.caffe_classes import class_names

In [ ]:
def fc(x, num_in, num_out, name, relu = True,stdv=0.01):
    with tf.variable_scope(name) as scope:
    
        # Create tf variables for the weights and biases
        weights = tf.get_variable('weights', shape=[num_in, num_out], trainable=True,
                              initializer = tf.truncated_normal_initializer(stddev=stdv))
        biases = tf.get_variable('biases', [num_out], trainable=True,
                             initializer = tf.constant_initializer(0.000))
    
        # Matrix multiply weights and inputs and add bias
        act = tf.nn.xw_plus_b(x, weights, biases, name=scope.name)
    
        if relu == True:
            # Apply ReLu non linearity
            relu = tf.nn.relu(act)      
            return relu
        else:
            return act
    



In [ ]:
"""
Predefine all necessary layer for the AlexNet
""" 
def conv(x, filter_height, filter_width, num_filters, stride_y, stride_x, name,
         padding='SAME', groups=1,weights_std = 0.1, bias_value = 0.001):
    """
      Adapted from: https://github.com/ethereon/caffe-tensorflow
    """
    # Get number of input channels
    input_channels = int(x.get_shape()[-1])
  
    # Create lambda function for the convolution
    convolve = lambda i, k: tf.nn.conv2d(i, k, 
                                       strides = [1, stride_y, stride_x, 1],
                                       padding = padding)
  
    with tf.variable_scope(name) as scope:
        # Create tf variables for the weights and biases of the conv layer
        shape = [filter_height, filter_width,
                              input_channels/groups, num_filters]
        weights = tf.get_variable('weights', shape = shape,
                                  initializer = tf.truncated_normal_initializer(stddev=weights_std))
        shape = [num_filters]
        biases = tf.get_variable('biases', shape = shape,
                                 initializer = tf.constant_initializer(value = bias_value))  
    
    
        if groups == 1:
            conv = convolve(x, weights)
            # In the cases of multiple groups, split inputs & weights and
        else:
            # Split input and weights and convolve them separately
            input_groups = tf.split(axis = 3, num_or_size_splits=groups, value=x)
            weight_groups = tf.split(axis = 3, num_or_size_splits=groups, value=weights)
            output_groups = [convolve(i, k) for i,k in zip(input_groups, weight_groups)]
      
            # Concat the convolved output together again
            conv = tf.concat(axis = 3, values = output_groups)
      
        # Add biases 
        bias = tf.reshape(tf.nn.bias_add(conv, biases), conv.get_shape().as_list())
    
        # Apply relu function
        relu = tf.nn.relu(bias, name = scope.name)
        
    return relu
  


In [ ]:
def max_pool(x, filter_height, filter_width, stride_y, stride_x, name, padding='SAME'):
    return tf.nn.max_pool(x, ksize=[1, filter_height, filter_width, 1],
                        strides = [1, stride_y, stride_x, 1],
                        padding = padding, name = name)
  


In [ ]:
def lrn(x, radius, alpha, beta, name, bias=1.0):
    return tf.nn.local_response_normalization(x, depth_radius = radius, alpha = alpha,
                                            beta = beta, bias = bias, name = name)
 

In [ ]:
 
def dropout(x, keep_prob):
    return tf.nn.dropout(x, keep_prob)
  

In [ ]:
inp = tf.placeholder(tf.float32,[1,256,256,3])
keep_prob = tf.placeholder(tf.float32,None)
num_classes = 1000


In [ ]:
with tf.variable_scope('alex_net') as sc:
     # 1st Layer: Conv (w ReLu) -> Pool -> Lrn
    conv1 = conv(inp, 11, 11, 96, 4, 4, padding = 'VALID', name = 'conv1')
    pool1 = max_pool(conv1, 3, 3, 2, 2, padding = 'VALID', name = 'pool1')
    norm1 = lrn(pool1, 2, 2e-05, 0.75, name = 'norm1')
    #self.conv1 = conv1
    # 2nd Layer: Conv (w ReLu) -> Pool -> Lrn with 2 groups
    conv2 = conv(norm1, 5, 5, 256, 1, 1, groups = 2, name = 'conv2')
    pool2 = max_pool(conv2, 3, 3, 2, 2, padding = 'VALID', name ='pool2')
    norm2 = lrn(pool2, 2, 2e-05, 0.75, name = 'norm2')
    #self.conv2 = conv2
    # 3rd Layer: Conv (w ReLu)
    conv3 = conv(norm2, 3, 3, 384, 1, 1, name = 'conv3')
    #self.conv3 = conv3
    # 4th Layer: Conv (w ReLu) splitted into two groups
    conv4 = conv(conv3, 3, 3, 384, 1, 1, groups = 2, name = 'conv4')
    #self.conv4 = conv4
    
    # 5th Layer: Conv (w ReLu) -> Pool splitted into two groups
    conv5 = conv(conv4, 3, 3, 256, 1, 1, groups = 2, name = 'conv5')
    #self.c5 = conv5
    pool5 = max_pool(conv5, 3, 3, 2, 2, padding = 'VALID', name = 'pool5')
    
    # 6th Layer: Flatten -> FC (w ReLu) -> Dropout
    flattened = tf.reshape(pool5, [-1, 6*6*256])
    conv5 = pool5
    fc6 = fc(flattened, 6*6*256, 4096, name='fc6')
    dropout6 = dropout(fc6, keep_prob)
    
    # 7th Layer: FC (w ReLu) -> Dropout
    fc7 = fc(dropout6, 4096, 4096, name = 'fc7')
    dropout7 = dropout(fc7, keep_prob)
    #self.fc7 = dropout7
    # 8th Layer: FC and return unscaled activations (for tf.nn.softmax_cross_entropy_with_logits)
    fc8 = fc(dropout7, 4096, num_classes, relu = False, name='fc8')
    sc.reuse_variables()

In [ ]:
for v in tf.trainable_variables():
    print(v)

In [ ]:
sess = tf.Session()
sess.run(tf.global_variables_initializer())

In [ ]:
for v in tf.trainable_variables():
    para = v.eval(sess)
    print(v.name,np.mean(para))

In [ ]:
import scipy.misc
img = scipy.misc.imread('/root/Documents/EEL881-2018/ell-881-2018-deep-learning/lec08/images/llama.jpeg')

In [ ]:
img.shape

In [ ]:
plt.imshow(img)

In [ ]:
o = sess.run(tf.argmax(tf.nn.softmax(fc8)),feed_dict={inp:img,keep_prob:1.0})

In [ ]:
img = scipy.misc.imresize(img,(256,256,3))

In [ ]:
img = img.reshape([1,256,256,3])

In [ ]:
o = sess.run(tf.argmax(tf.nn.softmax(fc8),axis=1),feed_dict={inp:img,keep_prob:1.0})

In [ ]:
o

In [ ]:
class_names[o[0]]

In [ ]:
o = sess.run([conv5],feed_dict={inp:img,keep_prob:1.0})

In [ ]:
o[0].shape

In [ ]:
data_dict = np.load('/root/Documents/EEL881-2018//bvlc_caffenet.npy',encoding = 'bytes').item()

In [ ]:
for op_name in data_dict:
    print(op_name)
    with tf.variable_scope('alex_net', reuse=True) :
        with tf.variable_scope(op_name,reuse=True) as sc:
            for param_name in data_dict[op_name]:
                print(param_name)
                try:
                    var = tf.get_variable(param_name.decode('utf-8'))
                    data = data_dict[op_name][param_name]
                    sess.run(var.assign(data))
                except ValueError:
                    raise    

sess.close()
sess = tf.Session()

for v in tf.trainable_variables():
    para = v.eval(sess)
    print(v.name,np.mean(para))
    

In [ ]:
o = sess.run(tf.argmax(tf.nn.softmax(fc8),axis=1),feed_dict={inp:img,keep_prob:1.0})
print(class_names[o[0]])

In [ ]:
img = scipy.misc.imread('/root/Documents/EEL881-2018/ell-881-2018-deep-learning/lec08/images/llama.jpeg')

In [ ]:
img = scipy.misc.imresize(img,(256,256,3))
img = img-np.array([104., 117., 124.])

In [ ]:

img = img.reshape([1,256,256,3])

o = sess.run(tf.argmax(tf.nn.softmax(fc8),axis=1),feed_dict={inp:img,keep_prob:1.0})

class_names[o[0]]

In [ ]:
img = scipy.misc.imread('/root/Documents/EEL881-2018/ell-881-2018-deep-learning/lec08/images/llama.jpeg')
img.shape

In [ ]:
img = img[32:256+32,70:256+70,:]
img = img-np.array([104., 117., 124.])
img.shape

In [ ]:
plt.imshow(img)

In [ ]:

img = img.reshape([1,256,256,3])

o = sess.run(tf.argmax(tf.nn.softmax(fc8),axis=1),feed_dict={inp:img,keep_prob:1.0})

class_names[o[0]]

In [ ]:
sess.close()

In [ ]:
def cross_entropy_loss(o,y):
    loss = tf.reduce_mean(tf.nn.softmax_cross_entropy_with_logits(logits = o, labels = y), name="cross_entropy_loss")
    return loss

In [ ]:
score = fc8
labels = tf.placeholder(tf.float32, [None, num_classes])

In [ ]:
ce_loss = cross_entropy_loss(score, labels)

In [ ]:
with tf.name_scope("train"):
    var_list = ft_var_list+train_var_list
    global_step = tf.Variable(0, trainable = False)
    learning_rate = tf.train.inverse_time_decay(parameters['learning_rate'], global_step,1, 0.0001)
    train_op,train_grads = op(loss,var_list,learning_rate,parameters['learning_rate_tr_mult'],global_step)
    gradients = train_grads

In [ ]:
GD = tf.train.gradinetDescentOP(learning_rate)
for epoch in range(parameters['start_epoch'],parameters['num_epochs']):
        print("{} Epoch number: {}".format(datetime.now(), epoch+1))
        step = 1
        sess.run(training_init_op)

        for step in range(train_batches_per_epoch):
            batch_x1s, batch_x2s, batch_x3s, batch_y1s, batch_y2s = sess.run(next_batch)

            sess.run(GD.mininlize(loss,var_list), feed_dict={x1: batch_x1s,
                                          x2: batch_x2s,
                                          x3: batch_x3s,
                                          y1: batch_y1s,
                                          y2: batch_y2s,
                                          keep_prob: parameters['dropout_rate']})

In [ ]:
opt = tf.train.GradientDescent(leafning_rate)


In [ ]:
def ops(loss,ft_var_list,train_var_list,learning_rate,lr_ft,lr_tr,global_step):
    opt_ft = tf.train.MomentumOptimizer(learning_rate*lr_ft,0.9)
    opt_train = tf.train.MomentumOptimizer(learning_rate*lr_tr,0.9)

    grads = tf.gradients(loss,ft_var_list+train_var_list)
    ft_grads = grads[:len(ft_var_list)]
    ft_grads = list(zip(ft_grads, ft_var_list))
    train_grads = grads[len(ft_var_list):]
    train_grads = list(zip(train_grads,train_var_list))

    if len(train_var_list)>0 and len(ft_var_list) > 0:
        ft_op = opt_ft.apply_gradients(grads_and_vars = ft_grads,global_step = global_step)
        train_op = opt_train.apply_gradients(grads_and_vars = train_grads, global_step = global_step)
        train_op = tf.group(ft_op, train_op)
    elif len(train_var_list)==0 and len(ft_var_list) > 0:
        ft_op = opt_ft.apply_gradients(grads_and_vars = ft_grads,global_step = global_step)
        train_op = ft_op
    elif len(train_var_list)>0 and len(ft_var_list) == 0:
        train_op = opt_train.apply_gradients(grads_and_vars = train_grads, global_step = global_step)

    return train_op,train_grads